## Variable Construction


In [8]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

#### UKPS Data

**Participation:**

- Breadth of cultural participation — the number of distinct cultural activities that a respondent engaged in over the past 12 months.
- Volume of cultural participation — the frequency of engagement across these activities, transformed into minimum annual counts for comparability.

- Intensity=Volume_norm × Breadth_norm

**Well‑being:**

- mean index; check Cronbach’s α

**Controls:**

- gender, age, education, occupation, ethnicity

In [ ]:
import pandas as pd

ukps_df = pd.read_csv('../data/UKDA-9350-tab/tab/participation_2023-24_annual_data_safeguard.tab', sep="\t")
list(ukps_df.columns)

['ArchYcserial',
 'ArchIndivSerial',
 'ArchHHSerial',
 'Year',
 'Quarter',
 'wave',
 'mode',
 'SubSample',
 'ScreenReader',
 'QRCODE',
 'NUMADULTS',
 'SEX',
 'COHAB',
 'CHILDHH',
 'CARTS1_001',
 'CARTS1_002',
 'CARTS1_003',
 'CARTS1_004',
 'CARTS1_005',
 'CARTS1_006',
 'CARTS1_007',
 'CARTS1_008',
 'CARTS1_009',
 'CARTS1_010',
 'CARTS1_011',
 'CARTS1_012',
 'CARTS1_013',
 'CARTS1_998',
 'CARTS1_996',
 'CARTS1A_a',
 'CARTS1A_b',
 'CARTS1A_c',
 'CARTS1A_d',
 'CARTS1A_e',
 'CARTS1A_f',
 'CARTS1A_g',
 'CARTS1A_h',
 'CARTS1A_i',
 'CARTS1A_j',
 'CARTS1A_k',
 'CARTS1A_l',
 'CARTS1A_m',
 'CARTS1B_a_001',
 'CARTS1B_a_002',
 'CARTS1B_a_003',
 'CARTS1B_a_999',
 'CARTS1B_b_001',
 'CARTS1B_b_002',
 'CARTS1B_b_003',
 'CARTS1B_b_999',
 'CARTS1B_c_001',
 'CARTS1B_c_002',
 'CARTS1B_c_003',
 'CARTS1B_c_999',
 'CARTS1B_d_001',
 'CARTS1B_d_002',
 'CARTS1B_d_003',
 'CARTS1B_d_999',
 'CARTS1B_e_001',
 'CARTS1B_e_002',
 'CARTS1B_e_003',
 'CARTS1B_e_999',
 'CARTS1B_f_001',
 'CARTS1B_f_002',
 'CARTS1B_f_003',


In [ ]:
ukps_df.ArchYcserial.nunique() == len(ukps_df)

True

### Participation

In [3]:
import re

binary_phys_part = set([x for x in ukps_df.columns if re.search("^CARTS[12]_0", x)])
freq_phys_part = set([x for x in ukps_df.columns if re.search("^CARTS[12]A_[a-z]", x)])

binary_digi_part = set([x for x in ukps_df.columns if re.search("^CARTS[34]_0", x)])
freq_digi_part = set([x for x in ukps_df.columns if re.search("^CARTS[34]A_[a-z]", x)])

recode_freq = {
    -5: pd.NA,
    -4: pd.NA,
    -3: pd.NA,
    1: 52,
    2: 12,
    3: 3,
    4: 2,
    5: 1,
    999: pd.NA,
}

ukps_df_participation = ukps_df.assign(
    phys_breath = ukps_df.loc[:, list(binary_phys_part)].gt(0).sum(axis=1),
    digi_breath = ukps_df.loc[:, list(binary_digi_part)].gt(0).sum(axis=1),
    phys_depth = ukps_df.loc[:, list(freq_phys_part)].replace(recode_freq).sum(axis=1),
    digi_depth = ukps_df.loc[:, list(freq_digi_part)].replace(recode_freq).sum(axis=1),
)[['ArchYcserial', 'phys_breath', 'digi_breath', 'phys_depth', 'digi_depth']]

ukps_df_participation.head()

,ArchYcserial,phys_breath,digi_breath,phys_depth,digi_depth
0,31000061,4,7,7,186
1,31000071,2,1,13,3
2,31000081,11,12,92,216
3,31000281,5,9,9,197
4,31000282,9,9,30,139


### Well-being

In [4]:
wellbeing_pos = set([x for x in ukps_df.columns if re.search("^WELLB[123]$", x)])

recode_wellb = {
    -5: pd.NA,
    -4: pd.NA,
    -3: pd.NA,
    997: pd.NA,
}

ukps_df_wellbeing = ukps_df.assign(
    WELLB4_rev = 10-ukps_df['WELLB4'].replace(recode_wellb),
    wellbeing = lambda x: x.loc[:, list(wellbeing_pos)+['WELLB4_rev']].replace(recode_wellb).mean(axis=1),
)[['ArchYcserial', 'wellbeing']]

ukps_df_wellbeing.head()

,ArchYcserial,wellbeing
0,31000061,6.5
1,31000071,7.5
2,31000081,7.25
3,31000281,3.25
4,31000282,7.0


In [ ]:
### TBC check alpha

### Control

In [5]:
control_vars = ['SEX', 'AGEBAND', 'NSSEC_5', 'EDUCAT4', 'ETHNIC_NET_Safeguard'] #gender age, education, occupation, ethnicity (only white non-white?)

recode_ctrl = {
    -5: pd.NA,
    -4: pd.NA,
    -3: pd.NA,
    997: pd.NA,
    999: pd.NA,
}

ukps_df_controls = ukps_df[['ArchYcserial']+control_vars].replace(recode_ctrl)

ukps_df_controls.head()

,ArchYcserial,SEX,AGEBAND,NSSEC_5,EDUCAT4,ETHNIC_NET_Safeguard
0,31000061,1,4,1,1,1
1,31000071,2,5,1,7,1
2,31000081,2,3,<NA>,1,1
3,31000281,2,3,1,7,1
4,31000282,1,2,1,4,1


### Final DF

In [9]:
from data.ukps_data_dict import lad23cd_codes

ukps_vars_df = ukps_df_participation.merge(
    ukps_df_wellbeing, how='left', on='ArchYcserial'
).merge(
    ukps_df_controls, how='left', on='ArchYcserial'
).merge(
    ukps_df.assign(LAD23CD = ukps_df.lad23cd.map(lad23cd_codes))[['ArchYcserial', 'LAD23CD']], how='left', on='ArchYcserial'
)

ukps_vars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171748 entries, 0 to 171747
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   ArchYcserial          171748 non-null  int64 
 1   phys_breath           171748 non-null  int64 
 2   digi_breath           171748 non-null  int64 
 3   phys_depth            171748 non-null  object
 4   digi_depth            171748 non-null  object
 5   wellbeing             168254 non-null  object
 6   SEX                   168801 non-null  object
 7   AGEBAND               168928 non-null  object
 8   NSSEC_5               147855 non-null  object
 9   EDUCAT4               121965 non-null  object
 10  ETHNIC_NET_Safeguard  165475 non-null  object
 11  LAD23CD               171748 non-null  object
dtypes: int64(3), object(9)
memory usage: 15.7+ MB
